# Capítulo 6 — Activos: contribución riesgo–retorno

**Objetivo:** ubicar cada ticker en el mapa volatilidad–CAGR y relacionarlo con su peso.


### Estadísticos por activo

In [1]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path("..").resolve()
sys.path.insert(0, str(ROOT / "src"))
import portfolio_utils as pu

daily = pd.read_csv(ROOT/"data"/"precios_ajustados_diarios.csv", index_col=0, parse_dates=True)
monthly = pu.to_month_end(daily)
px = monthly.loc["2015-12-31":"2026-09-30"]
rows=[]
for t,w in pu.CURRENT_WEIGHTS.items():
    s = px[t].dropna()
    # retornos desde ene-2016
    r = s.pct_change().loc["2016-01-31":].dropna()
    years = len(r)/12
    # valor de 1 unidad
    v0 = s.loc[: "2015-12-31"].iloc[-1]
    vT = s.loc["2016-01-31":"2026-09-30"].iloc[-1]
    cagr = (vT/v0)**(1/years) - 1
    vol = r.std(ddof=1)*np.sqrt(12)
    rows.append({"ticker":t, "peso":w, "cagr":cagr, "vol":vol})
adf = pd.DataFrame(rows)
print(adf.assign(peso_pct=adf.peso*100, cagr_pct=adf.cagr*100, vol_pct=adf.vol*100)
        [["ticker","peso_pct","cagr_pct","vol_pct"]].round(2).to_string(index=False))


ticker  peso_pct  cagr_pct  vol_pct
  SPYG     30.51     17.23    17.06
   SMH     21.76     34.96    28.39
 BRK-B     20.40     13.14    17.61
  IEMG     20.15      9.78    16.96
   VTI      7.18     14.45    15.50


### Scatter riesgo–retorno

In [2]:
fig, ax = plt.subplots(figsize=(8,6))
ax.scatter(adf.vol*100, adf.cagr*100, s=adf.peso*2500, alpha=0.7)
for _,row in adf.iterrows():
    ax.annotate(row.ticker, (row.vol*100, row.cagr*100), xytext=(6,4), textcoords="offset points")
ax.set_xlabel("Volatilidad anualizada (%)")
ax.set_ylabel("CAGR (%)")
ax.set_title("Riesgo–retorno por activo (tamaño = peso eToro)")
plt.tight_layout()
plt.savefig(ROOT/"figures"/"06_activos_riesgo_retorno.png", dpi=120)
plt.show()


![figura](../figures/06_activos_riesgo_retorno.svg)


**Interpretación:** `SMH` aporta gran parte del motor de retorno (y del riesgo). `IEMG` diversifica geográficamente pero históricamente diluyó el CAGR frente a un 100% EE.UU. growth.

## Conclusiones
- El tamaño del bubble chart recuerda que el peso actual ya incorpora el *drift* de mercado (SMH creció mucho).
- VTI es un satélite pequeño (~7%).

## Ejercicios propuestos
1. Calcula la contribución aproximada al riesgo $w_i \cdot \mathrm{Cov}(R_i,R_p)/\sigma_p$.
2. ¿Qué pasa con el CAGR del portafolio si pones IEMG en 0% y redistribuyes a SPYG?
3. Compara BRK-B vs SPYG solo en 2022.
